# NB14 — Posterior cache, stage-1 model (SPEC T2)

Runs the stage-1 model (`checkpoint-5000`) **once** on Extra-dev, fold A, fold B and the Iqra dev slice, and saves
the raw float32 frame logits (log-posteriors are computed on load; float16 storage flips near-tied decodes,
SPEC decision 26), greedy decodes and emission spans. Everything in T3 runs on these files.
Scores the stage-1 greedy decode on Extra-dev with the official scorer: **baseline B0 (ladder row 7)**.
Never touches QuranMB.v2 or IqraEval_Test_GT.

**Run settings:** Accelerator **GPU T4** (one is enough), Internet *On*.
Add data: `iqra-cache`, the NB11 output (`iqra-splits` or the NB11 notebook's output), `quran-phoneme-model` (holds `checkpoint-5000`),
and the NB8 model output (notebook `notebook17aed3d165`, folder `quran_model_final`; found by its weights hash,
used only for the transformers version-parity check, decision 27).
Secrets: `HF_TOKEN`; `GITHUB_TOKEN` only if the repo is private.

**After the run:** Save Version, then *Create Dataset* from the output, named `iqra-post-stage1`.
Paste back everything printed by the last cell (**PASTE BACK**).


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Environment: one GPU, Kaggle's own transformers (decision 27), versions --------
# transformers 4.44.2 (used by NB2/NB8) cannot be installed on Python 3.13 (no tokenizers
# 0.19 wheel). We use Kaggle's installed version and prove below that it computes the
# same thing as 4.44.2 before running anything.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")      # single GPU, as NB2 (DataParallel bug)
import time, json, random
import numpy as np
import pandas as pd
import torch, transformers, datasets
from transformers import AutoFeatureExtractor
from iqra import data, parity, posteriors, scorer
from iqra.candidates import remove_insertions
from iqra.config import Config, find_cache_root, find_input
from iqra.decode import greedy_decode
from iqra.vocab import Vocab

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
versions = {"python": sys.version.split()[0], "torch": torch.__version__,
            "transformers": transformers.__version__, "datasets": datasets.__version__,
            "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"}
print(versions)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    raise RuntimeError("HF_TOKEN secret is required for the gated Iqra_Extra_IS26") from e


In [ ]:
# ---- Version parity with transformers 4.44.2 (decision 27): must pass ---------------
# The trained NB8 model (model-3) is run on the reference inputs on CPU, so any
# difference comes from the library version, not from GPU arithmetic.
meta_ref, ref = parity.load_fixture()
# Find the NB8 model by its weights hash, whatever folder it is mounted under.
NB8_FINAL, checked = parity.find_reference_model(meta_ref["weights_sha256"])
for path, h in checked:
    print(f"  {h[:12]}  {path}")
print("NB8 reference model:", NB8_FINAL)
m_cpu, info = parity.load_with_report(NB8_FINAL, "cpu")
assert not any(info.values()), f"NB8 model did not load cleanly: {info}"
fe_ref = AutoFeatureExtractor.from_pretrained(NB8_FINAL)
real = {k[: -len("_features_input")]: v for k, v in ref.items() if k.endswith("_features_input")}
cur_cpu = parity.run(m_cpu, fe_ref, "cpu", real)
ok, rows = parity.compare(ref, cur_cpu)
for r in rows:
    print(r)
print(f"reference: transformers {meta_ref['transformers']}, torch {meta_ref['torch']} | "
      f"here: transformers {transformers.__version__}, torch {torch.__version__} | PARITY {'PASS' if ok else 'FAIL'}")
gpu_rows = None
if DEVICE == "cuda":            # informational: same library, GPU vs CPU arithmetic
    cur_gpu = parity.run(m_cpu.to(DEVICE), fe_ref, DEVICE, real)
    gpu_rows = [{"input": r["input"], "gpu_vs_cpu_max_abs_logit_diff": float(np.max(np.abs(cur_gpu[k] - cur_cpu[k]))),
                 "gpu_vs_cpu_argmax_mismatch_frames": int(np.sum(cur_gpu[k].argmax(-1) != cur_cpu[k].argmax(-1)))}
                for r in rows for k in [r["input"] + "_logits"]]
    for r in gpu_rows:
        print(r)
del m_cpu
assert ok, "transformers on this image does not reproduce 4.44.2: stop and report (fallback: option B)"
version_parity = {"pass": ok, "rows": rows, "gpu_vs_cpu": gpu_rows, "reference": {
    k: meta_ref[k] for k in ("transformers", "torch")}}


In [ ]:
# ---- Inputs: cache, splits, checkpoint, vocab --------------------------------------
cfg = Config()
cfg.paths.cache_root = find_cache_root()
SPLITS_DIR = find_input("splits.json", what="iqra-splits (the NB11 output dataset)")
CKPT = find_input("checkpoint-5000/config.json", what="quran-phoneme-model (holds checkpoint-5000)")
OUT = "/kaggle/working/nb14_stage1"
os.makedirs(OUT, exist_ok=True)
print("iqra-cache:", cfg.paths.cache_root, "\nsplits:", SPLITS_DIR, "\ncheckpoint:", CKPT)

vocab = Vocab.load(cfg.paths.vocab)
ID2TOK, BLANK = vocab.id2tok, vocab.blank_id
splits = data.load_splits(f"{SPLITS_DIR}/splits.json")
truth_dev = pd.read_csv(f"{SPLITS_DIR}/truth_extra_dev.csv", keep_default_na=False)
report = {"repo_commit": REPO_COMMIT, "versions": versions, "checkpoint": CKPT,
          "splits_commit": splits.get("repo_commit"), "version_parity": version_parity}

model, load_info = parity.load_with_report(CKPT, DEVICE)
assert not any(load_info.values()), f"checkpoint-5000 did not load cleanly: {load_info}"
try:
    fe = AutoFeatureExtractor.from_pretrained(CKPT)
    fe_src = CKPT
except Exception:
    fe = AutoFeatureExtractor.from_pretrained("facebook/w2v-bert-2.0")   # as NB2 did
    fe_src = "facebook/w2v-bert-2.0"
FRAME_RATE = posteriors.model_frame_rate(model.config)
assert model.config.vocab_size == len(vocab.tok2id), (model.config.vocab_size, len(vocab.tok2id))
assert model.config.pad_token_id == BLANK, (model.config.pad_token_id, BLANK)
print(f"model: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params | vocab {len(vocab.tok2id)} "
      f"(hash {vocab.hash}) | blank {BLANK} | frame rate {FRAME_RATE}/s | feature extractor: {fe_src}")
report.update({"vocab_hash": vocab.hash, "frame_rate": FRAME_RATE, "feature_extractor": fe_src})


In [ ]:
# ---- Audio sources for each split, IDs verified against splits.json ---------------
from datasets import load_dataset, load_from_disk
extra = load_dataset(cfg.datasets.extra, split="train", token=HF_TOKEN)
ex_ids = [str(x) for x in list(extra["id"])]
assert ex_ids == splits["extra"]["ids_by_row"], "Iqra_Extra_IS26 row order changed since NB11"
row_of = {cid: r for r, cid in enumerate(ex_ids)}

dev = load_from_disk(cfg.paths.train_arrow)["dev"]
slice_rows = splits["iqra_dev_slice"]["rows"]
dev_ids_now = [str(x) for x in list(dev["id"])]
assert [dev_ids_now[r] for r in slice_rows] == splits["iqra_dev_slice"]["ids"], "Iqra dev slice changed"

SOURCES = {   # split -> (dataset, rows, ids)
    "extra_dev": (extra, [row_of[i] for i in splits["extra"]["extra_dev"]], splits["extra"]["extra_dev"]),
    "fold_a": (extra, [row_of[i] for i in splits["extra"]["fold_a"]], splits["extra"]["fold_a"]),
    "fold_b": (extra, [row_of[i] for i in splits["extra"]["fold_b"]], splits["extra"]["fold_b"]),
    "iqra_dev_slice": (dev, slice_rows, splits["iqra_dev_slice"]["ids"]),
}
for k, (_, rows, ids) in SOURCES.items():
    print(f"{k:15s} {len(rows)} clips")


In [ ]:
# ---- Run the model once per split; write the caches --------------------------------
cache_stats = {}
for split, (ds, rows, ids) in SOURCES.items():
    t0 = time.time()
    entries, rates, frames, secs, fp16_mismatch = {}, set(), 0, 0.0, 0
    sub = ds.select(rows)
    for k, arr, sr in data.iter_audio(sub):
        rates.add(sr)
        audio = data.to_16k(arr, sr)
        logits = posteriors.clip_logits(model, fe, audio, DEVICE)
        entry = posteriors.make_entry(logits, ID2TOK, BLANK)          # raw float32 logits
        fp16_mismatch += posteriors.float16_would_flip(entry, ID2TOK, BLANK)
        entries[ids[k]] = entry
        frames += logits.shape[0]
        secs += len(audio) / 16_000
    path = f"{OUT}/post_stage1_{split}.pt"
    posteriors.write_cache(path, entries, checkpoint="stage1-checkpoint-5000", split=split,
                           vocab_hash=vocab.hash, frame_rate=FRAME_RATE,
                           extra_meta={"repo_commit": REPO_COMMIT, "source_rates": sorted(rates)})
    cache_stats[split] = {"clips": len(entries), "hours": round(secs / 3600, 3),
                          "source_sample_rates": sorted(rates),
                          "measured_frames_per_s": round(frames / secs, 2),
                          "clips_float16_would_flip": int(fp16_mismatch),
                          "mb": round(os.path.getsize(path) / 1e6, 1), "minutes": round((time.time() - t0) / 60, 1)}
    print(split, cache_stats[split])
report["caches"] = cache_stats


In [ ]:
# ---- Done-when 1: cached decode == live decode on 20 random clips -----------------
def nb5_recognise(logits):          # NB5's recognise_phonemes, from logits onward
    out, prev = [], None
    for i in logits.argmax(-1).tolist():
        if i != prev and i != BLANK:
            sym = ID2TOK[i] if i < len(ID2TOK) else "<unk>"
            if sym not in ("<sil>", "sil", "<pad>", "<unk>"):
                out.append(sym)
        prev = i
    return " ".join(out)

rng = random.Random(13)
picks = rng.sample([(s, i) for s, (_, _, ids) in SOURCES.items() for i in range(len(ids))], 20)
loaded = {s: posteriors.read_cache(f"{OUT}/post_stage1_{s}.pt")[1] for s in SOURCES}
mismatch = []
for split, i in picks:
    ds, rows, ids = SOURCES[split]
    _, arr, sr = next(data.iter_audio(ds.select([rows[i]])))
    live = nb5_recognise(posteriors.clip_logits(model, fe, data.to_16k(arr, sr), DEVICE))
    cached = " ".join(posteriors.decode_entry(loaded[split][ids[i]], ID2TOK, BLANK)[0])
    if live != cached:
        mismatch.append((split, ids[i]))
print("live vs cached decode, 20 random clips: mismatches =", len(mismatch), mismatch[:5])
assert not mismatch, "cache does not reproduce the live decode"
report["live_check"] = {"clips": 20, "mismatches": len(mismatch)}


In [ ]:
# ---- Done-when 2: baseline B0 on Extra-dev (official scorer) + sentence bootstrap -
def predictions(split, transform=None):
    rows = []
    ref = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"]))
    for cid, entry in loaded[split].items():
        toks = posteriors.decode_entry(entry, ID2TOK, BLANK)[0]
        if transform:
            toks = transform(ref[cid].split(), toks)
        rows.append({"ID": cid, "Prediction": " ".join(toks)})
    return pd.DataFrame(rows)

pred_b0 = predictions("extra_dev")
empty_preds = pred_b0[pred_b0["Prediction"].str.strip() == ""]
print("empty predictions on Extra-dev:", len(empty_preds))
pred_b0.to_csv(f"{OUT}/pred_b0_extra_dev.csv", index=False)
b0 = scorer.official_score(truth_dev, pred_b0, keep_dir=f"{OUT}/score_b0_extra_dev")
counts_b0 = scorer.per_clip_counts(truth_dev, pred_b0)
counts_b0.to_csv(f"{OUT}/counts_b0_extra_dev.csv")
clusters = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"]))
ci = scorer.bootstrap_f1_interval(counts_b0, clusters)
assert int(counts_b0["TR"].sum()) == b0["TR"] and int(counts_b0["FR"].sum()) == b0["FR"]
print(f"B0 (stage-1, greedy) on Extra-dev: P={b0['precision']} R={b0['recall']} F1={b0['f1']} "
      f"PER={b0['per']} | 90% CI over {ci['n_clusters']} sentences: [{ci['lo']:.4f}, {ci['hi']:.4f}]")
print(f"counts: TR={b0['TR']} FR={b0['FR']} FA={b0['FA']} TA={b0['TA']} (per-clip sums match)")
report["B0"] = {k: b0[k] for k in ("precision", "recall", "f1", "per", "TR", "FR", "FA", "TA")}
report["B0"]["ci90"] = [round(ci["lo"], 4), round(ci["hi"], 4)]
report["B0"]["n_sentences"] = ci["n_clusters"]


In [ ]:
# ---- Decision 22 check: insertions kept vs all insertions removed (report only) ----
pred_noins = predictions("extra_dev", transform=remove_insertions)
noins = scorer.official_score(truth_dev, pred_noins)
gain = scorer.paired_bootstrap_gain(counts_b0, scorer.per_clip_counts(truth_dev, pred_noins), clusters)
n_ins = sum(len(a.split()) - len(b.split()) for a, b in zip(pred_b0["Prediction"], pred_noins["Prediction"]))
print(f"insertions in B0 decodes: {n_ins} | F1 kept {b0['f1']} vs removed {noins['f1']} "
      f"(gain {gain['gain']:+.4f}, 90% CI [{gain['lo']:+.4f}, {gain['hi']:+.4f}])")
if noins["f1"] > b0["f1"]:
    print(">>> REPORT: removing insertions scored higher on Extra-dev (not switching; decision 22)")
report["insertion_check"] = {"insertions": n_ins, "f1_kept": b0["f1"], "f1_removed": noins["f1"],
                             "gain_removed": round(gain["gain"], 4), "ci90": [round(gain["lo"], 4), round(gain["hi"], 4)]}


In [ ]:
# ---- Quranic share (fixed matcher, rasm comparison) and B0 by subset ---------------
try:
    quran = data.load_tanzil_simple_clean()
    texts = [str(x) for x in list(extra["sentence"])]
    flags = data.quranic_flags(texts, quran)
    share = {k: round(float(np.mean([flags[r] for r in rows])), 3)
             for k, (ds, rows, _) in SOURCES.items() if ds is extra}
    print("Quranic share (lower bound):", share)
    print("not matched, examples:", list(dict.fromkeys(texts[r] for r in range(len(texts)) if not flags[r]))[:5])
    dev_flag = {cid: flags[row_of[cid]] for cid in truth_dev["ID"]}
    for name, keep in (("Quranic", True), ("not matched", False)):
        sub = counts_b0[[dev_flag[i] == keep for i in counts_b0.index]]
        if len(sub):
            m = scorer.summed_metrics(sub)
            print(f"  B0 on {name} Extra-dev clips ({len(sub)}): F1={m['f1']:.4f} P={m['precision']:.4f} R={m['recall']:.4f}")
except Exception as e:
    share = f"unknown: {type(e).__name__}: {e}"
    print("Quranic share", share)
report["quranic_share"] = share


In [ ]:
# ---- Correct speech: what the stage-1 model flags on the Iqra dev slice ------------
from iqra.align import accuracy_counts
dev_phon = list(dev["phoneme_ref"])
dev_ref = {i: str(dev_phon[r]) for i, r in zip(splits["iqra_dev_slice"]["ids"], slice_rows)}
edits = total = flagged_clips = 0
for cid, entry in loaded["iqra_dev_slice"].items():
    ref = dev_ref[cid].split()
    e, n = accuracy_counts(ref, posteriors.decode_entry(entry, ID2TOK, BLANK)[0])
    edits += e; total += n; flagged_clips += e > 0
print(f"Iqra dev slice (correct speech): PER {edits / total:.4f} | clips with any flag: "
      f"{flagged_clips}/{len(dev_ref)} | edits = verifier negatives available: {edits}")
report["iqra_dev_slice"] = {"per": round(edits / total, 4), "clips_flagged": flagged_clips,
                            "edits": edits, "clips": len(dev_ref)}
with open(f"{OUT}/nb14_stage1_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1, default=str)


In [ ]:
# ---- PASTE BACK ----------------------------------------------------------------------
print(json.dumps(report, ensure_ascii=False, indent=1, default=str))
print("files:", sorted(os.listdir(OUT)))
